# 🐾 ระบบดึงข้อมูลสายพันธุ์สัตว์เลี้ยงลูกด้วยนม (Mammalia Species Data Retrieval)

สมุดบันทึกนี้ออกแบบมาสำหรับใช้งานบน **Google Colab** เพื่อดึงข้อมูลสายพันธุ์สัตว์เลี้ยงลูกด้วยนม (Mammals) จากฐานข้อมูลระดับโลก **GBIF (Global Biodiversity Information Facility) API** ซึ่งใช้งานได้ฟรีโดยไม่ต้องมี API Key

---

### 1. ติดตั้งและนำเข้าไลบรารีที่จำเป็น (Import Libraries)

In [1]:
import pandas as pd
import numpy as np
import requests
import os
import json
import shutil
import random
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
# from google.colab import files # สำหรับดาวน์โหลดไฟล์บน Colab


# 1. เชื่อมต่อกับ Google Drive (ยกเลิก Comment เมื่อรันบน Colab จริง)
from google.colab import drive
drive.mount('/content/drive')

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))


Mounted at /content/drive
TensorFlow Version: 2.20.0
GPU Available: []


### 2. โหลดข้อมูลสปีชีส์จากไฟล์ JSON (Load Species Data from JSON)

เราจะโหลดข้อมูลสายพันธุ์สัตว์เลี้ยงลูกด้วยนมจากไฟล์ JSON ที่เราอัปโหลดไว้บน Google Drive

In [2]:
# กำหนดเส้นทางไฟล์ใน Google Drive
json_path = "/content/drive/MyDrive/Colab Notebooks/SmartZoo/data/species/mammal-0039340-260623161305970.json"

if not os.path.exists(json_path):
    print(f"❌ ไม่พบไฟล์ที่ตำแหน่ง: {json_path}")
    print("กรุณาตรวจสอบว่าได้อัปโหลดไฟล์ไปที่ Google Drive ของคุณในตแหน่งที่ถูกต้องแล้ว")
else:
    with open(json_path, "r", encoding="utf-8") as f:
        species_list = json.load(f)
    print(f"✅ โหลดข้อมูลสำเร็จ: พบสายพันธุ์ทั้งหมด {len(species_list)} รายการ")
    
    # แสดงตัวอย่างข้อมูล 3 รายการแรก
    print(json.dumps(species_list[:3], ensure_ascii=False, indent=2))

✅ โหลดข้อมูลสำเร็จ: พบสายพันธุ์ทั้งหมด 12788 รายการ
[
  {
    "taxonKey": 2312877,
    "scientificName": "Fimbriosthenelais marianae Lana, 1991",
    "acceptedTaxonKey": 2312877,
    "acceptedScientificName": "Fimbriosthenelais marianae Lana, 1991",
    "numberOfOccurrences": 37,
    "taxonRank": "SPECIES",
    "taxonomicStatus": "ACCEPTED",
    "kingdom": "Animalia",
    "kingdomKey": 1,
    "phylum": "Annelida",
    "phylumKey": 42,
    "class": "Polychaeta",
    "classKey": 256,
    "order": "Phyllodocida",
    "orderKey": 1079,
    "family": "Sigalionidae",
    "familyKey": 7081,
    "genus": "Fimbriosthenelais",
    "genusKey": 2312871,
    "species": "Fimbriosthenelais marianae",
    "speciesKey": 2312877,
    "iucnRedListCategory": null
  },
  {
    "taxonKey": 8231708,
    "scientificName": "Loimia armata Carrerette & Nogueira, 2015",
    "acceptedTaxonKey": 8231708,
    "acceptedScientificName": "Loimia armata Carrerette & Nogueira, 2015",
    "numberOfOccurrences": 24,
    "t

### 3. ตรวจสอบจำนวนภาพใน GBIF API เพื่อใช้สำหรับเทรน (Query GBIF API for Image Counts)

เราจะทำการวนลูปเพื่อยิง API ไปถาม GBIF ว่าแต่ละ `taxonKey` มีรูปภาพ (StillImage) ในฐานข้อมูลกี่ภาพ สำหรับเตรียมนำมาใช้ในการดาวน์โหลดภาพไปเทรนโมเดลต่อไป

> [!NOTE]
> การเรียก API ทั้งหมด 12,788 รายการอาจใช้เวลานาน (ประมาณ 30-40 นาที) เราจึงแนะนำให้ทำการกรอง (Filter) เฉพาะกลุ่มเป้าหมายหรือดึงเฉพาะสปีชีส์ที่มีบันทึกจำนวนมากก่อน หรือสามารถรันตัวอย่างทดสอบ 100 รายการแรกเพื่อดูผลลัพธ์ได้

In [6]:
from tqdm.notebook import tqdm
import time
import os

# 1. คัดกรองเบื้องต้นเพื่อลดจำนวนการเรียก API
# คัดเลือกเฉพาะระดับ SPECIES หรือ SUBSPECIES และมีจำนวน Occurrence >= 10 ขึ้นไป
filtered_species = [
    sp for sp in species_list 
    if sp.get("taxonRank") in ["SPECIES", "SUBSPECIES"] 
    and sp.get("numberOfOccurrences") is not None 
    and sp.get("numberOfOccurrences") >= 100
]

print(f"จำนวนสายพันธุ์ที่ผ่านการคัดกรองเบื้องต้น: {len(filtered_species)} จากทั้งหมด {len(species_list)} รายการ")

# 2. ฟังก์ชันตรวจสอบจำนวนภาพ (StillImage) ใน GBIF API
def get_gbif_image_count(taxon_key):
    # ระบุอีเมลผู้ติดต่อเพื่อความสุภาพกับ API Server ปลายทาง
    headers = {
        "User-Agent": "SmartZooResearchBot/1.0 (your-email@example.com)"
    }
    url = f"https://api.gbif.org/v1/occurrence/search?taxonKey={taxon_key}&mediaType=StillImage&limit=0"
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            return response.json().get("count", 0)
    except Exception as e:
        pass
    return 0

# 3. ตรวจสอบไฟล์สำรองเดิมที่มีอยู่
output_json_path = "/content/drive/MyDrive/Colab Notebooks/SmartZoo/data/species/mammal_with_image_counts.json"
processed_data = {}

if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r", encoding="utf-8") as f:
            existing_list = json.load(f)
            # โหลดคีย์เดิมที่เคยสำเร็จแล้วเก็บไว้ใน Dictionary
            processed_data = {item["taxonKey"]: item for item in existing_list if "taxonKey" in item}
        print(f"📊 ตรวจพบข้อมูลเดิมที่ดึงสำเร็จแล้ว {len(processed_data)} รายการ")
    except Exception as e:
        print(f"⚠️ ไม่สามารถโหลดไฟล์สำรองเดิมได้เนื่องจาก: {e} (จะเริ่มดึงข้อมูลใหม่ทั้งหมด)")

# 4. กรองเอาเฉพาะรายชื่อสายพันธุ์ที่ยังไม่เคยดึงข้อมูล
# ตั้งค่าขีดจำกัดเพื่อทดสอบรัน (เปลี่ยนเป็น None หากต้องการรันทั้งหมดโดยไม่มีขีดจำกัด)
test_limit = None
species_to_query = [sp for sp in filtered_species if sp.get("taxonKey") not in processed_data]

if test_limit:
    species_to_query = species_to_query[:test_limit]
    print(f"(จำกัดการรันเพื่อทดสอบที่ {test_limit} รายการแรก)")

print(f"คงเหลือสายพันธุ์ที่ต้องดึงข้อมูลเพิ่ม: {len(species_to_query)} รายการ")

# 5. เริ่มดึงข้อมูลพร้อมระบบสำรองผลลัพธ์ทุก ๆ 1,000 รายการ
counter = 0

if len(species_to_query) > 0:
    print("กำลังดึงข้อมูลจำนวนภาพ...")
    for sp in tqdm(species_to_query):
        taxon_key = sp.get("taxonKey")
        if taxon_key:
            image_count = get_gbif_image_count(taxon_key)
            sp["gbifImageCount"] = image_count
        else:
            sp["gbifImageCount"] = 0
            
        # บันทึกสถานะลงตัวแปรกลาง
        processed_data[taxon_key] = sp
        counter += 1
        
        # บันทึกไฟล์สำรองอัตโนมัติทุก ๆ 1,000 รายการ
        if counter % 1000 == 0:
            with open(output_json_path, "w", encoding="utf-8") as f:
                json.dump(list(processed_data.values()), f, ensure_ascii=False, indent=2)
            print(f"💾 สำรองข้อมูลอัตโนมัติสำเร็จ ณ รายการที่ {counter} (รวมในไฟล์สำรองทั้งหมด {len(processed_data)} รายการ)")
            
        # หน่วงเวลา 0.1 วินาทีต่อคำขอ เพื่อถนอม API server
        time.sleep(0.1)

    # บันทึกข้อมูลทั้งหมดเป็นครั้งสุดท้ายเมื่อรันเสร็จสิ้น
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(list(processed_data.values()), f, ensure_ascii=False, indent=2)
    print(f"\n✅ การดึงข้อมูลเสร็จสิ้นสมบูรณ์! บันทึกไฟล์ใหม่เรียบร้อยที่: {output_json_path} (รวมสะสมทั้งหมด {len(processed_data)} รายการ)")
else:
    print("🎉 ดึงข้อมูลสายพันธุ์ครบทั้งหมดเรียบร้อยแล้ว ไม่จำเป็นต้องรันเพิ่ม")

# 6. แสดงตัวอย่าง 5 สายพันธุ์ที่มีภาพมากที่สุด
sorted_results = sorted(list(processed_data.values()), key=lambda x: x.get("gbifImageCount", 0), reverse=True)
print("\n📊 ตัวอย่าง 5 สายพันธุ์ที่มีรูปภาพมากที่สุดในฐานข้อมูล GBIF:")
for i, sp in enumerate(sorted_results[:5], 1):
    print(f"{i}. {sp['scientificName']} (TaxonKey: {sp['taxonKey']}) -> มีภาพจัดเก็บอยู่ {sp.get('gbifImageCount', 0)} ภาพ")

จำนวนสายพันธุ์ที่ผ่านการคัดกรองเบื้องต้น: 552 จากทั้งหมด 12788 รายการ
📊 ตรวจพบข้อมูลเดิมที่ดึงสำเร็จแล้ว 0 รายการ
คงเหลือสายพันธุ์ที่ต้องดึงข้อมูลเพิ่ม: 552 รายการ
กำลังดึงข้อมูลจำนวนภาพ...


  0%|          | 0/552 [00:00<?, ?it/s]


✅ การดึงข้อมูลเสร็จสิ้นสมบูรณ์! บันทึกไฟล์ใหม่เรียบร้อยที่: /content/drive/MyDrive/Colab Notebooks/SmartZoo/data/species/mammal_with_image_counts.json (รวมสะสมทั้งหมด 552 รายการ)

📊 ตัวอย่าง 5 สายพันธุ์ที่มีรูปภาพมากที่สุดในฐานข้อมูล GBIF:
1. Anas platyrhynchos Linnaeus, 1758 (TaxonKey: 9761484) -> มีภาพจัดเก็บอยู่ 850437 ภาพ
2. Apis mellifera Linnaeus, 1758 (TaxonKey: 1341976) -> มีภาพจัดเก็บอยู่ 712958 ภาพ
3. Passer domesticus (Linnaeus, 1758) (TaxonKey: 5231190) -> มีภาพจัดเก็บอยู่ 460490 ภาพ
4. Vanessa atalanta (Linnaeus, 1758) (TaxonKey: 1898286) -> มีภาพจัดเก็บอยู่ 401610 ภาพ
5. Ardea cinerea Linnaeus, 1758 (TaxonKey: 9797180) -> มีภาพจัดเก็บอยู่ 330150 ภาพ
